# Tutorial 2: QuadraSHAP for product-kernel models

This notebook explains one prediction from an RBF `KernelRidge` model. We will:

1. compute exact local Shapley values with QuadraSHAP;
2. enumerate every feature coalition as a transparent exact baseline;
3. verify attribution agreement and additivity;
4. choose the number of quadrature nodes *a priori* for a prescribed accuracy;
5. switch between the neutral-factor, baseline and empirical interventional value functions; and
6. compare execution times.

> Run this notebook from an environment where the repository has been installed with `python -m pip install -e .`.

## Why an RBF kernel is a product game

For a fitted dual model, `f(x) = sum_r alpha[r] * k(x, X_train[r])`. The RBF kernel factorizes across features:

`k(x, z) = product_j exp(-gamma * (x[j] - z[j])**2)`.

For coalition `S`, the first part of this tutorial includes only factors whose features are in `S`; missing factors are replaced by the multiplicative identity `1`. This is the **neutral-factor** value function of Section 4 of the paper (the value function of PKeX-Shapley), and its empty-coalition value for `KernelRidge` is `sum(alpha)`. The baseline and empirical interventional value functions, which replace absent factors by those of a reference point or a background dataset, are covered towards the end of the notebook; all three are weighted sums of product games and share the same quadrature.

## From an exponential Shapley sum to one integral

For $d$ features, the Shapley value of feature $i$ is

$$
\phi_i(v)=\sum_{S\subseteq D\setminus\{i\}}
\underbrace{\frac{|S|!(d-|S|-1)!}{d!}}_{\mu(|S|)}
\left[v(S\cup\{i\})-v(S)\right].
$$

For a product game $v(S)=\prod_{j\in S}u_j$, the marginal contribution becomes $(u_i-1)\prod_{j\in S}u_j$. Proposition 2 of the paper uses the Beta-function identity

$$
\mu(s)=\int_0^1 t^s(1-t)^{d-s-1}\,dt
$$

to absorb every factorial weight into an integral. Once the coalition sum is moved inside, each feature $j\ne i$ contributes $(1-t)$ when absent and $tu_j$ when present. Summing every possible choice expands the product

$$
\boxed{\phi_i(v)=(u_i-1)\int_0^1\prod_{j\ne i}[(1-t)+tu_j]\,dt.}
$$

This is the conceptual heart of QuadraSHAP: a sum over $2^{d-1}$ subsets becomes a one-dimensional integral of a degree-$(d-1)$ polynomial.

## Why Gauss-Legendre quadrature is exact

An $m_q$-node Gauss-Legendre rule on $[0,1]$ exactly integrates every polynomial through degree $2m_q-1$. Since the leave-one-feature-out product above has degree at most $d-1$,

$$2m_q-1\ge d-1\quad\Longrightarrow\quad m_q\ge\lceil d/2\rceil$$

is sufficient for exact Shapley values up to floating-point arithmetic (Proposition 3). Below this threshold, the paper proves a geometric error bound of the form $C_\rho M_\rho |u_i-1|\rho^{-2m_q}$ for any Bernstein-ellipse parameter $\rho>1$, and the appendix on choosing the number of nodes makes every constant explicit, so that the smallest certified budget for a tolerance $\varepsilon$ can be computed from the factor tables *before* any quadrature is run. By default `QuadraSHAP` uses that budget with $\varepsilon=10^{-3}$; `m_q="exact"` selects the threshold $\lceil d/2\rceil$.

In [ ]:
import math
import time

import numpy as np
from sklearn.datasets import make_regression
from sklearn.kernel_ridge import KernelRidge

from quadrashap import RKHSExplainer

np.set_printoptions(precision=6, suppress=True)

## Train a small RBF kernel model

Fourteen features give the naive implementation `2**14 = 16,384` coalitions. QuadraSHAP needs only `ceil(14 / 2) = 7` quadrature nodes for exactness.

In [2]:
N_FEATURES = 14
N_TRAIN = 160
GAMMA = 0.2
M_Q_EXACT = (N_FEATURES + 1) // 2

X, y = make_regression(
    n_samples=N_TRAIN + 20,
    n_features=N_FEATURES,
    n_informative=8,
    noise=0.1,
    random_state=1,
)
X_train, y_train = X[:N_TRAIN], y[:N_TRAIN]
x = X[N_TRAIN]

model = KernelRidge(kernel="rbf", gamma=GAMMA, alpha=1.0).fit(X_train, y_train)
print(f"training rows={N_TRAIN}, features={N_FEATURES}, coalitions={2**N_FEATURES:,}")

training rows=160, features=14, coalitions=16,384


## From one product game to the fitted kernel model

Each training row $r$ defines its own product game with factors

$$u_j^{(r)}=\exp[-\gamma(x_j-X_{rj})^2].$$

The kernel-specific coalition value from Section 4.1 of the paper is

$$v_x(S)=\sum_{r=1}^n\alpha_r\prod_{j\in S}u_j^{(r)},\qquad v_x(\varnothing)=\sum_r\alpha_r.$$

Shapley values are linear, so QuadraSHAP explains every row-level product game and combines the answers with the learned dual coefficients:

$$
\phi_i(v_x)=\sum_{r=1}^n\alpha_r(u_i^{(r)}-1)
\sum_{q=1}^{m_q}\omega_q\prod_{j\ne i}[1-\tau_q+\tau_q u_j^{(r)}].
$$

This is why the explainer needs the model's training/support points, dual coefficients, and factorized per-feature kernel.

In [3]:
raw_nodes, raw_weights = np.polynomial.legendre.leggauss(M_Q_EXACT)
quad_nodes = 0.5 * (raw_nodes + 1.0)
quad_weights = 0.5 * raw_weights

print(f"exact node count: {M_Q_EXACT}")
print("nodes on [0, 1]:  ", np.round(quad_nodes, 6))
print("weights (sum = 1):", np.round(quad_weights, 6))

exact node count: 7
nodes on [0, 1]:   [0.025446 0.129234 0.297077 0.5      0.702923 0.870766 0.974554]
weights (sum = 1): [0.064742 0.139853 0.190915 0.20898  0.190915 0.139853 0.064742]


## Explain with QuadraSHAP

We first request the exact rule, `m_q="exact"`, i.e. `ceil(d / 2)` nodes, so that the result can be compared with exhaustive enumeration. The NumPy log-space backend is portable and avoids constructing the larger prefix/suffix tensor.

In [ ]:
explainer = RKHSExplainer(model, backend="logspace_numpy")
phi_quadra = explainer.explain(x, m_q="exact")

# For KernelRidge there is no intercept, and every empty product equals 1.
baseline = explainer.value_function_at_empty("neutral")   # == sum(alpha)
prediction = float(model.predict(x[None, :])[0])
reconstruction = baseline + phi_quadra.sum()

print(f"quadrature nodes: {M_Q_EXACT}")
print(f"baseline:         {baseline: .8f}")
print(f"sum(phi):         {phi_quadra.sum(): .8f}")
print(f"prediction:       {prediction: .8f}")
print(f"reconstruction:   {reconstruction: .8f}")

## Sharing work across features

A literal evaluation would rebuild a product of $d-1$ terms for every pair `(feature, quadrature node)`, costing $O(d^2m_q)$ for one product game. QuadraSHAP defines

$$T_{q,j}=1-\tau_q+\tau_q u_j,\qquad P_q=\prod_{j=1}^dT_{q,j}.$$

Every leave-one-out product is then $P_q/T_{q,i}$. Computing each full product once and reusing it reduces the work to $O(dm_q)$. For a model containing $n$ weighted product games, the total is $O(ndm_q)$, or $O(nd^2)$ at the exact threshold - still polynomial instead of exponential.

The library provides two ways to obtain these leave-one-out products:

- **prefix/suffix scan:** multiply all terms before and after each feature, avoiding division and mapping naturally to associative parallel scans;
- **shared log-space product:** accumulate $\log|P_q|$ and the sign, then subtract $\log|T_{q,i}|$ for each omitted factor.

The second approach prevents long products from overflowing or underflowing. Explicit sign tracking keeps it valid for negative factors; exact zeros require a separate case, as discussed in Section 3.2. With enough processors, the associative reductions have $O(\log d)$ parallel depth (Proposition 4). This notebook selects `logspace_numpy`; the JAX methods expose the scan-friendly implementations.

## Watch the quadrature converge

The exact threshold is a guarantee, not always the smallest useful budget. Here is the error from each smaller node count relative to the exact result, next to the certified bound $A_{\max}\,B(m_q,\Lambda_{\max})$ of the appendix (computed from the factor tables alone) and the efficiency residual $|\sum_i\phi_i-(f(x)-v(\varnothing))|$, which is exactly zero at the threshold and a cheap necessary check below it.

In [ ]:
from quadrashap.product_games.budget import certify

summary = explainer.summarize(x, "neutral")   # A_i and Lambda_max of this instance's product games
print(f"Lambda_max = {summary.lambda_max:.3f},  A_max = {summary.A_max:.4f}")
print(f"{'m_q':>4} {'max |phi - phi_exact|':>24} {'certified bound':>18} {'efficiency residual':>22}")
for m_q in range(1, M_Q_EXACT + 1):
    phi_mq = explainer.explain(x, m_q=m_q)
    err = np.abs(phi_mq - phi_quadra).max()
    residual = abs(baseline + phi_mq.sum() - prediction)
    print(f"{m_q:4d} {err:24.3e} {certify(summary, m_q):18.3e} {residual:22.3e}")

## Choosing the number of nodes a priori

`explain` with the default `m_q=None` computes the certified budget for the tolerance `eps` (default $10^{-3}$) and returns it in a report when asked. The certificate bounds the absolute quadrature error of every attribution; the observed error is typically two to three orders of magnitude smaller. With only 14 features the exactness threshold is already 7 nodes, so the saving is small here; at $d=1000$ the same procedure certifies a few tens of nodes against a threshold of 500 (see `benchmarks/kernel_node_budget_bench.py`).

In [ ]:
phi_auto, report = explainer.explain(x, return_report=True)
print(report)
print(f"observed max |phi - phi_exact| = {np.abs(phi_auto - phi_quadra).max():.3e}  <=  certified {report.bound:.3e}  <=  eps {report.eps:g}")

for eps in (1e-2, 1e-4, 1e-8):
    rep = explainer.node_budget(x, eps=eps)
    print(f"eps = {eps:6.0e} -> {rep.m_q:2d} nodes (exactness threshold {rep.exact_threshold})")

## A naive exact baseline

The implementation below evaluates the dual model for all feature subsets and then applies the standard factorial Shapley weights. It is intentionally direct rather than optimized.

In [6]:
def naive_kernel_shapley(model, x, gamma):
    """Exact product-game Shapley values by exhaustive enumeration."""
    X_train = np.asarray(model.X_fit_, dtype=float)
    alpha = np.asarray(model.dual_coef_, dtype=float).reshape(-1)
    d = X_train.shape[1]

    # factors[r, j] is the one-dimensional RBF kernel for row r, feature j.
    factors = np.exp(-gamma * (X_train - x[None, :]) ** 2)
    coalition_values = np.empty(1 << d, dtype=float)
    coalition_values[0] = alpha.sum()

    for mask in range(1, 1 << d):
        selected = [j for j in range(d) if mask & (1 << j)]
        coalition_kernel = factors[:, selected].prod(axis=1)
        coalition_values[mask] = alpha @ coalition_kernel

    factorial = [math.factorial(k) for k in range(d + 1)]
    phi = np.zeros(d, dtype=float)
    for feature in range(d):
        feature_bit = 1 << feature
        for mask in range(1 << d):
            if mask & feature_bit:
                continue
            size = mask.bit_count()
            weight = factorial[size] * factorial[d - size - 1] / factorial[d]
            phi[feature] += weight * (
                coalition_values[mask | feature_bit] - coalition_values[mask]
            )

    return phi, coalition_values[0]

In [7]:
phi_naive, baseline_naive = naive_kernel_shapley(model, x, GAMMA)

np.testing.assert_allclose(phi_quadra, phi_naive, rtol=1e-8, atol=1e-8)
np.testing.assert_allclose(baseline, baseline_naive, rtol=1e-12, atol=1e-12)
np.testing.assert_allclose(reconstruction, prediction, rtol=1e-9, atol=1e-9)

print(f"maximum attribution difference: {np.max(np.abs(phi_quadra - phi_naive)):.3e}")
print(f"additivity error:               {abs(reconstruction - prediction):.3e}")

print("\nFive largest attributions:")
for feature in np.argsort(np.abs(phi_quadra))[::-1][:5]:
    print(f"  feature {feature:2d}: {phi_quadra[feature]: .6f}")

maximum attribution difference: 1.023e-12
additivity error:               2.007e-13

Five largest attributions:
  feature 11: -341.464058
  feature  7:  308.522626
  feature  0:  82.085815
  feature 10:  80.062888
  feature  6:  67.993049


## Baseline and empirical interventional value functions

The same machinery serves the other value functions of Section 4. For a reference point $x^b$ the **baseline** value function is $v(S)=f(x_S, x^b_{-S})$: the absent factors become $k_j(x^b_j, X_{rj})$ instead of $1$. Averaging over the rows of a background dataset gives the **empirical interventional** value function; its Shapley values are the average of the baseline ones and its cost grows linearly with the number of background rows. Efficiency now reads $\sum_i\phi_i = f(x)-f(x^b)$ and $\sum_i\phi_i = f(x)-\tfrac1{n_b}\sum_b f(\tilde x^b)$ respectively.

In [ ]:
x_baseline = X_train.mean(axis=0)
background = X_train[:25]

phi_base, rep_base = explainer.explain(x, "baseline", baseline=x_baseline, return_report=True)
phi_int, rep_int = explainer.explain(x, "interventional", background=background, return_report=True)

print(f"baseline:       {rep_base.m_q} nodes, sum(phi) = {phi_base.sum(): .6f}, "
      f"f(x) - f(x_b) = {prediction - model.predict(x_baseline[None, :])[0]: .6f}")
print(f"interventional: {rep_int.m_q} nodes, sum(phi) = {phi_int.sum(): .6f}, "
      f"f(x) - mean_b f(x_b) = {prediction - model.predict(background).mean(): .6f}")

phi_avg = np.mean([explainer.explain(x, "baseline", baseline=b, m_q="exact") for b in background], axis=0)
print(f"interventional == mean of baselines: {np.abs(phi_avg - explainer.explain(x, 'interventional', background=background, m_q='exact')).max():.2e}")

print("\nTop-5 features under each value function:")
for name, phi in (("neutral", phi_quadra), ("baseline", phi_base), ("interventional", phi_int)):
    top = np.argsort(np.abs(phi))[::-1][:5]
    print(f"  {name:15s} {top}  {np.round(phi[top], 4)}")

## Timing comparison

Both timed calls include construction of the per-feature RBF factors for the observation. Model fitting and explainer construction are outside the timed region.

In [ ]:
def timed(callable_, repeats):
    durations = []
    result = None
    for _ in range(repeats):
        start = time.perf_counter()
        result = callable_()
        durations.append(time.perf_counter() - start)
    return result, float(np.median(durations))


_ = explainer.explain(x, m_q="exact")
_, quadra_exact_seconds = timed(lambda: explainer.explain(x, m_q="exact"), repeats=20)
_, quadra_budget_seconds = timed(lambda: explainer.explain(x), repeats=20)   # a priori budget, incl. its selection
_, naive_seconds = timed(lambda: naive_kernel_shapley(model, x, GAMMA), repeats=3)

print(f"QuadraSHAP, exact rule:      {quadra_exact_seconds * 1e3:9.3f} ms")
print(f"QuadraSHAP, certified budget:{quadra_budget_seconds * 1e3:9.3f} ms")
print(f"Naive enumeration:           {naive_seconds * 1e3:9.3f} ms")
print(f"Measured speedup (exact):    {naive_seconds / quadra_exact_seconds:9.1f}x")

## Why the gap grows

The exact naive method evaluates `2**d` coalitions. QuadraSHAP replaces that subset enumeration with `ceil(d / 2)` Gauss–Legendre nodes. For this model, the resulting integrand is a polynomial, so that quadrature order is exact up to floating-point precision.

In [9]:
print(f"{'features':>8} {'naive coalitions':>20} {'exact quadrature nodes':>24}")
for d in [10, 14, 20, 30]:
    print(f"{d:8d} {2**d:20,d} {(d + 1) // 2:24d}")

features     naive coalitions   exact quadrature nodes
      10                1,024                        5
      14               16,384                        7
      20            1,048,576                       10
      30        1,073,741,824                       15


## Beyond this toy comparison

The paper's experiments study both the quadrature budget and competitive kernel baselines at much larger scales:

- Section 5.1 fits RBF Kernel Ridge models with $d\in\{50,500,1000,5000\}$ and measures mean $\ell_2$ error over ten test points. Although exactness requires $\lceil d/2\rceil$ nodes, the observed error falls rapidly with much smaller budgets; for example, at $d=1000$ it falls from roughly 70 at $m_q=20$ to roughly $10^{-4}$ at $m_q=50$, versus an exactness threshold of 500.
- Section 5.3 compares with the exact PKeX-Shapley baseline over 50 held-out instances per synthetic setting and a 300-second per-instance timeout. The paper reports a 25.8x speedup at $d=500$ and 95.5x at $d=1000$; PKeX-Shapley times out at $d\ge2000$, while QuadraSHAP completes at $d=2000$ and $d=5000$ using at most 400 nodes.

Those are controlled paper benchmarks against a specialized polynomial-time baseline. The timing above is deliberately simpler: it compares against the exponential definition so that correctness and the source of the scaling improvement are easy to inspect.

## Takeaway

The exhaustive and quadrature calculations agree, but they reach the result in very different ways. Enumeration is ideal for understanding and testing tiny examples; quadrature is the practical route once the feature count grows. The displayed speedup is measured on your machine and will vary with hardware and package versions.